# Student Performance Prediction
## Step 4: Prepare Data for Modeling

**Course:** CS4661 – Introduction to Data Science
**Group members:** Eric Castellon, Kalen Hoang, Royce Jamerson, Alexander Woeste, Rojina Zalzar
**Assigned owner for this task:** Eric Castellon

### Notebook objectives
1. Load the dataset and the proposed scope from Steps 1–2 (`project_outputs/project_scope.json`).
2. Re-check that the selected inputs and the GPA target are ready for modeling.
3. Document the type of each input and the preprocessing decision for it.
4. Create one reproducible train/test split that every model will share.
5. Define a preprocessing pipeline that is fitted on the training data only.
6. Save the split and the preprocessing settings for the modeling step.

This notebook does **not** train or evaluate models. That is the next task (Implement selected models).

**Scope status:** This notebook follows the scope proposed in Steps 1–2, which is still pending group approval. If the group changes the target or the inputs, update `project_scope.json` and rerun this notebook.

### Setup
Run this notebook from the repository's `notebooks/` folder after running `01_Dataset_Review_and_EDA.ipynb`. It reads the CSV from `data/` and the scope from `project_outputs/`, writes `src/preprocessing.py`, and saves its files to `project_outputs/modeling/`.

## Environment setup
Uncomment the installation command only if an import fails.

In [1]:
# Run this command only if an import fails.
# %pip install pandas scikit-learn

from pathlib import Path
import json
import platform

import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from IPython.display import display

DATA_PATH = Path("../data/Student_performance_data _.csv")
OUTPUT_DIR = Path("../project_outputs")
SCOPE_PATH = OUTPUT_DIR / "project_scope.json"

# Save this step's files in their own folder.
MODELING_DIR = OUTPUT_DIR / "modeling"
MODELING_DIR.mkdir(parents=True, exist_ok=True)

print("Python version:", platform.python_version())
print("pandas version:", pd.__version__)
print("scikit-learn version:", sklearn.__version__)

Python version: 3.13.9
pandas version: 2.2.3
scikit-learn version: 1.6.1


# 4A. Load the data and the proposed scope
The target and input variables are read from the scope file saved in Step 2 instead of being typed again, so a change to the scope only needs to be made in one place. The original CSV is loaded fresh; nothing depends on variables from the Steps 1–2 notebook.

In [2]:
for path in [DATA_PATH, SCOPE_PATH]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Cannot find {path}. Run 01_Dataset_Review_and_EDA.ipynb first "
            "and run this notebook from the notebooks/ folder."
        )

with open(SCOPE_PATH, encoding="utf-8") as file:
    scope = json.load(file)

TARGET = scope["target"]
FEATURES = scope["features"]
EXCLUDED_COLUMNS = scope["excluded_columns"]
ID_COLUMN = "StudentID"
SUBGROUP_COLUMNS = ["Gender", "Ethnicity"]

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

missing_columns = sorted(
    set(FEATURES + [TARGET, ID_COLUMN] + SUBGROUP_COLUMNS) - set(df.columns)
)
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

assert TARGET not in FEATURES, "The target must not be an input."
assert not set(FEATURES).intersection(EXCLUDED_COLUMNS), \
    "An excluded column is listed as an input."

print("Scope status:", scope["status"])
print("Task:", scope["task_type"])
print("Target:", TARGET)
print("Inputs:", FEATURES)
print("Data shape:", df.shape)

Scope status: Proposed — pending group approval
Task: Regression
Target: GPA
Inputs: ['Age', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering']
Data shape: (2392, 15)


# 4B. Re-check the data before splitting
Steps 1–2 found no missing values, duplicates or invalid GPA values. These checks are repeated here so this notebook stops if the CSV has changed. They also confirm that every coded variable only contains the codes described in the source documentation.

No rows are removed or changed. If a check fails, the group should decide how to handle the affected records before continuing.

In [3]:
# Expected codes, from the variable definitions reviewed in Step 1.
EXPECTED_CODES = {
    "Age": {15, 16, 17, 18},
    "ParentalEducation": {0, 1, 2, 3, 4},
    "ParentalSupport": {0, 1, 2, 3, 4},
    "Tutoring": {0, 1},
    "Extracurricular": {0, 1},
    "Sports": {0, 1},
    "Music": {0, 1},
    "Volunteering": {0, 1},
}

checks = {
    "Missing values in inputs or target":
        int(df[FEATURES + [TARGET]].isna().sum().sum()),
    "Exact duplicate rows": int(df.duplicated().sum()),
    "Repeated student IDs": int(df[ID_COLUMN].duplicated().sum()),
    "GPA outside 0–4": int((~df[TARGET].between(0, 4)).sum()),
    "Negative weekly study time": int((df["StudyTimeWeekly"] < 0).sum()),
    "Negative absences": int((df["Absences"] < 0).sum()),
}
for column, codes in EXPECTED_CODES.items():
    checks[f"Unexpected codes in {column}"] = int((~df[column].isin(codes)).sum())

check_table = pd.Series(checks, name="Count").to_frame()
display(check_table)

if check_table["Count"].sum() > 0:
    raise ValueError("Resolve the flagged data issues before preparing the data.")
print("All checks passed. No rows were removed or changed.")

,Count
Missing values in inputs or target,0
Exact duplicate rows,0
Repeated student IDs,0
GPA outside 0–4,0
Negative weekly study time,0
Negative absences,0
Unexpected codes in Age,0
Unexpected codes in ParentalEducation,0
Unexpected codes in ParentalSupport,0
Unexpected codes in Tutoring,0


All checks passed. No rows were removed or changed.


# 4C. Choose a preprocessing method for each input
Each input is assigned to one group, and each group gets one preprocessing decision.

| Group | Variables | Decision | Reason |
|---|---|---|---|
| Numerical | Age, StudyTimeWeekly, Absences | Standardize (mean 0, SD 1) | The variables use very different scales (hours, counts, years). Standardizing puts Linear Regression coefficients on a comparable scale. Tree models are not affected by scaling. |
| Ordered categories | ParentalEducation, ParentalSupport | Keep the ordered codes 0–4, then standardize | The levels have a natural order (None → Higher / Very High), so the codes keep that order with one column each. |
| Yes/No | Tutoring, Extracurricular, Sports, Music, Volunteering | Keep as 0/1, no scaling | They are already numeric indicators; a 1 means the student has that attribute. |

**Not needed:** No imputation (there are no missing values) and no one-hot encoding (no unordered category is used as an input).

**Points for the group to review:**
- In Step 3, average GPA did not rise steadily across ParentalEducation levels (levels 3–4 were lower than 1–2). Treating it as one ordered column assumes a steady trend for Linear Regression. One-hot encoding it is a reasonable alternative if the group prefers.
- Absences had a very strong correlation with GPA in Step 3 (r = −0.92). The modeling step should also report results without Absences so the comparison shows what the other inputs add. The saved settings include that alternative input list.

In [4]:
NUMERICAL_FEATURES = ["Age", "StudyTimeWeekly", "Absences"]
ORDINAL_FEATURES = ["ParentalEducation", "ParentalSupport"]
BINARY_FEATURES = [
    "Tutoring", "Extracurricular", "Sports", "Music", "Volunteering",
]

# Every selected input must belong to exactly one group.
grouped = NUMERICAL_FEATURES + ORDINAL_FEATURES + BINARY_FEATURES
assert sorted(grouped) == sorted(FEATURES), \
    "Feature groups do not match the inputs in project_scope.json."
assert len(grouped) == len(set(grouped)), "A feature is listed in two groups."

feature_plan = pd.DataFrame(
    [(f, "Numerical", "Standardize") for f in NUMERICAL_FEATURES]
    + [(f, "Ordered category (0–4)", "Keep codes, standardize") for f in ORDINAL_FEATURES]
    + [(f, "Yes/No (0/1)", "Keep as is") for f in BINARY_FEATURES],
    columns=["Variable", "Type", "Preprocessing"],
)
display(feature_plan)
feature_plan.to_csv(MODELING_DIR / "feature_preprocessing_plan.csv", index=False)

,Variable,Type,Preprocessing
0,Age,Numerical,Standardize
1,StudyTimeWeekly,Numerical,Standardize
2,Absences,Numerical,Standardize
3,ParentalEducation,Ordered category (0–4),"Keep codes, standardize"
4,ParentalSupport,Ordered category (0–4),"Keep codes, standardize"
5,Tutoring,Yes/No (0/1),Keep as is
6,Extracurricular,Yes/No (0/1),Keep as is
7,Sports,Yes/No (0/1),Keep as is
8,Music,Yes/No (0/1),Keep as is
9,Volunteering,Yes/No (0/1),Keep as is


# 4D. Split the data into training and testing sets
- **Split:** 80% training, 20% testing.
- **Random seed:** 42, so the split is the same every time the notebook runs.
- **Stratification:** GPA is grouped into five equal-sized bins (quintiles) and the split keeps the same share of each bin in both sets. This keeps the test set's GPA distribution close to the training set's, so no model is judged on an unusually easy or hard test set.

The test set is set aside for the final model comparison only. All preprocessing and model tuning must use the training set.

In [5]:
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_GPA_BINS = 5

# GPA bins are used only to balance the split; they are not an input.
gpa_bins = pd.qcut(df[TARGET], q=N_GPA_BINS, labels=False)

train_idx, test_idx = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=gpa_bins,
)

train_df = df.loc[train_idx].sort_index()
test_df = df.loc[test_idx].sort_index()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

# No student may appear in both sets.
overlap = set(train_df[ID_COLUMN]) & set(test_df[ID_COLUMN])
assert not overlap, f"{len(overlap)} students appear in both sets."
assert len(train_df) + len(test_df) == len(df)

print("Training set:", X_train.shape, "| Testing set:", X_test.shape)

# Compare the GPA distribution of the two sets.
gpa_comparison = pd.DataFrame({
    "Training": y_train.describe(),
    "Testing": y_test.describe(),
}).round(3)
display(gpa_comparison)

Training set: (1913, 10) | Testing set: (479, 10)


,Training,Testing
count,1913.000,479.000
mean,1.906,1.905
std,0.913,0.926
min,0.000,0.000
25%,1.175,1.186
50%,1.895,1.886
75%,2.616,2.644
max,4.000,4.000


# 4E. Build the preprocessing pipeline
The preprocessing steps are defined once in `src/preprocessing.py`. The modeling notebook imports the same function, so every model uses exactly the same preprocessing.

The preprocessor is combined with each model in a scikit-learn `Pipeline`. When the pipeline is fitted, the scaler learns its means and standard deviations from the **training data only**, and the same values are then applied to the test data. This prevents information from the test set leaking into training.

In [6]:
%%writefile ../src/preprocessing.py
"""Shared preprocessing for the Student Performance Prediction project.

Created in Step 4 (Prepare data for modeling). Import build_preprocessor()
in the modeling notebook and combine it with each model in a Pipeline so
that scaling is fitted on the training data only.
"""
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NUMERICAL_FEATURES = ["Age", "StudyTimeWeekly", "Absences"]
ORDINAL_FEATURES = ["ParentalEducation", "ParentalSupport"]
BINARY_FEATURES = [
    "Tutoring", "Extracurricular", "Sports", "Music", "Volunteering",
]


def build_preprocessor(features=None):
    """Return an unfitted ColumnTransformer for the selected inputs.

    features: optional list of input names to keep (for example, the
    inputs without Absences). Defaults to all ten selected inputs.
    """
    def keep(columns):
        return [c for c in columns if features is None or c in features]

    return ColumnTransformer(
        transformers=[
            ("scaled", StandardScaler(),
             keep(NUMERICAL_FEATURES) + keep(ORDINAL_FEATURES)),
            ("binary", "passthrough", keep(BINARY_FEATURES)),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

Overwriting ../src/preprocessing.py


In [7]:
import importlib
import sys

# Make the shared code in src/ importable.
SRC_DIR = str(Path("../src").resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import preprocessing
importlib.reload(preprocessing)  # Pick up edits to preprocessing.py on rerun.
from preprocessing import build_preprocessor

# Check the preprocessing groups match this notebook's groups.
assert preprocessing.NUMERICAL_FEATURES == NUMERICAL_FEATURES
assert preprocessing.ORDINAL_FEATURES == ORDINAL_FEATURES
assert preprocessing.BINARY_FEATURES == BINARY_FEATURES

# Demonstration only: fit on the training set and transform both sets.
preprocessor = build_preprocessor()
X_train_prepared = pd.DataFrame(
    preprocessor.fit_transform(X_train),
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index,
)
X_test_prepared = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index,
)

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared testing shape:", X_test_prepared.shape)

# Leakage check: scaled training columns have mean 0 and SD 1 exactly;
# the test columns are close but not exact, because the test set was
# transformed with the training set's statistics.
scaled_columns = NUMERICAL_FEATURES + ORDINAL_FEATURES
leakage_check = pd.DataFrame({
    "TrainMean": X_train_prepared[scaled_columns].mean(),
    "TrainSD": X_train_prepared[scaled_columns].std(ddof=0),
    "TestMean": X_test_prepared[scaled_columns].mean(),
    "TestSD": X_test_prepared[scaled_columns].std(ddof=0),
}).round(3)
display(leakage_check)

print("Prepared training preview:")
display(X_train_prepared.head().round(3))

Prepared training shape: (1913, 10)
Prepared testing shape: (479, 10)


,TrainMean,TrainSD,TestMean,TestSD
Age,-0.0,1.0,0.094,0.986
StudyTimeWeekly,-0.0,1.0,0.020,0.963
Absences,0.0,1.0,-0.003,1.033
ParentalEducation,-0.0,1.0,0.014,0.993
ParentalSupport,0.0,1.0,0.015,0.980


Prepared training preview:


,Age,StudyTimeWeekly,Absences,ParentalEducation,ParentalSupport,Tutoring,Extracurricular,Sports,Music,Volunteering
0,0.491,1.771,-0.897,0.256,-0.105,1.0,0.0,0.0,1.0,0.0
2,-1.286,-0.973,1.362,1.255,-0.105,0.0,0.0,0.0,0.0,0.0
4,0.491,-0.892,0.292,0.256,0.782,1.0,0.0,0.0,0.0,0.0
5,1.379,-0.274,-1.730,-0.742,-0.993,0.0,1.0,0.0,0.0,0.0
6,-1.286,1.028,-0.541,-0.742,0.782,0.0,0.0,1.0,0.0,0.0


# 4F. Save the prepared data and settings
Files saved in `project_outputs/modeling/` (at the repository root):

| File | Contents |
|---|---|
| `train.csv` | StudentID, the 10 selected inputs, and GPA for the training set (unscaled) |
| `test.csv` | The same columns for the testing set (unscaled) |
| `subgroup_columns.csv` | StudentID, split, Gender and Ethnicity, kept separately for the possible subgroup review. Not model inputs. |
| `preprocessing_config.json` | Split settings, feature groups, preprocessing decisions and the alternative inputs without Absences |
| `feature_preprocessing_plan.csv` | The preprocessing decision for each input |

The CSVs are saved **unscaled** on purpose. Scaling is fitted inside each model's pipeline on the training data, so the saved files can be used with any model.

Rerunning this notebook overwrites these files. With the same CSV and seed, the split is identical.

In [8]:
KEEP_COLUMNS = [ID_COLUMN] + FEATURES + [TARGET]
train_df[KEEP_COLUMNS].to_csv(MODELING_DIR / "train.csv", index=False)
test_df[KEEP_COLUMNS].to_csv(MODELING_DIR / "test.csv", index=False)

subgroups = df[[ID_COLUMN] + SUBGROUP_COLUMNS].copy()
subgroups.insert(1, "Split", "train")
subgroups.loc[test_idx, "Split"] = "test"
subgroups.to_csv(MODELING_DIR / "subgroup_columns.csv", index=False)

FEATURES_WITHOUT_ABSENCES = [f for f in FEATURES if f != "Absences"]

preprocessing_config = {
    "project_name": scope["project_name"],
    "step": "Step 4 – Prepare data for modeling",
    "owner": "Eric Castellon",
    "status": "Prepared — scope pending group approval",
    "source_file": DATA_PATH.name,
    "target": TARGET,
    "features": FEATURES,
    "features_without_absences": FEATURES_WITHOUT_ABSENCES,
    "id_column": ID_COLUMN,
    "subgroup_columns": SUBGROUP_COLUMNS,
    "excluded_columns": EXCLUDED_COLUMNS,
    "split": {
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "stratified_on": f"GPA quantile bins (q={N_GPA_BINS})",
        "train_rows": len(train_df),
        "test_rows": len(test_df),
    },
    "feature_groups": {
        "numerical_standardized": NUMERICAL_FEATURES,
        "ordinal_codes_standardized": ORDINAL_FEATURES,
        "binary_passthrough": BINARY_FEATURES,
    },
    "preprocessing": {
        "missing_value_handling": "None needed (no missing values found)",
        "encoding": "None needed (ordered codes kept; binaries already 0/1)",
        "scaling": "StandardScaler on numerical and ordinal inputs, fitted on training data only",
        "rows_removed": 0,
        "shared_code": "src/preprocessing.py -> build_preprocessor()",
    },
    "rules_for_modeling": [
        "Use train.csv for fitting and tuning; use test.csv only for the final comparison.",
        "Combine build_preprocessor() with each model in a Pipeline.",
        "Do not use GPA, GradeClass, Gender or Ethnicity as inputs.",
        "Also report results using features_without_absences.",
    ],
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__,
    "prepared_date": pd.Timestamp.today().date().isoformat(),
}

with open(MODELING_DIR / "preprocessing_config.json", "w", encoding="utf-8") as file:
    json.dump(preprocessing_config, file, indent=2, ensure_ascii=False)

for path in sorted(MODELING_DIR.iterdir()):
    print("Saved:", path)

Saved: ../project_outputs/modeling/feature_preprocessing_plan.csv
Saved: ../project_outputs/modeling/preprocessing_config.json
Saved: ../project_outputs/modeling/subgroup_columns.csv
Saved: ../project_outputs/modeling/test.csv
Saved: ../project_outputs/modeling/train.csv


# 4G. Check the saved files
Reload the saved files the way the modeling notebook will, and confirm they match what was prepared above.

In [9]:
check_train = pd.read_csv(MODELING_DIR / "train.csv")
check_test = pd.read_csv(MODELING_DIR / "test.csv")

assert check_train.shape == (len(train_df), len(KEEP_COLUMNS))
assert check_test.shape == (len(test_df), len(KEEP_COLUMNS))
assert not set(check_train[ID_COLUMN]) & set(check_test[ID_COLUMN])
assert check_train[TARGET].round(10).equals(train_df[TARGET].reset_index(drop=True).round(10))
for column in ["GradeClass"] + SUBGROUP_COLUMNS:
    assert column not in check_train.columns

print("Saved files reload correctly.")
print("Training rows:", len(check_train), "| Testing rows:", len(check_test))

Saved files reload correctly.
Training rows: 1913 | Testing rows: 479


## How the modeling notebook should use these files
```python
import json
import sys
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
sys.path.insert(0, "../src")   # shared code lives in src/
from preprocessing import build_preprocessor

config = json.load(open("../project_outputs/modeling/preprocessing_config.json"))
train = pd.read_csv("../project_outputs/modeling/train.csv")
test = pd.read_csv("../project_outputs/modeling/test.csv")

features, target = config["features"], config["target"]
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

model = Pipeline([
    ("preprocess", build_preprocessor(features)),
    ("model", LinearRegression()),
])
model.fit(X_train, y_train)   # the scaler is fitted on training data only
predictions = model.predict(X_test)
```
For the comparison without Absences, use `config["features_without_absences"]` in place of `config["features"]`.

The mean-prediction baseline should use the **training** GPA mean (for example, `sklearn.dummy.DummyRegressor(strategy="mean")`).

### Step 4 review notes
- **Data checks:** The data checks found no missing values, duplicate rows, repeated student IDs, out-of-range GPA values or unexpected category codes. No rows were removed or changed.
- **Inputs and target:** The 10 inputs and the GPA target follow `project_scope.json` from Step 2. StudentID, GradeClass, Gender and Ethnicity are not used as inputs. Gender and Ethnicity are saved separately for the possible subgroup review.
- **Preprocessing:** Standardize Age, StudyTimeWeekly, Absences, ParentalEducation and ParentalSupport. Keep the five Yes/No variables as 0/1. No imputation or one-hot encoding is needed.
- **Split:** 80/20 train/test split, seed 42, stratified on GPA quintiles. It gives 1,913 training rows and 479 testing rows, with no student in both sets.
- **Leakage prevention:** Scaling is fitted inside each model's pipeline on the training data only. The test set is reserved for the final comparison.
- **For group review:**
  - Approve the scope from Step 2.
  - Decide whether ParentalEducation should stay as one ordered column or be one-hot encoded.
  - Agree to also report results without Absences.
- **Reviewer and date:** November 25, 2026 by Eric Castellon

**Completion criterion:** The group agrees on the preprocessing decisions and split, and the modeling notebook loads the saved files without changes.

## Progress tracker
| Task | Owner | Status | Evidence |
|---|---|---|---|
| Prepare data for modeling | Eric Castellon | Complete — pending group approval | Data checks, preprocessing plan, train/test split, `src/preprocessing.py`, saved files in `project_outputs/modeling/` |

## Next project task
Implement selected models (Royce Jamerson).